# Register dualview FOVs to the lowres overview

Registration stage only: each highres FOV is downsampled to the lowres voxel size and located in the lowres volume by phase cross correlation.
The result (translation of every FOV in lowres micrometer coordinates) is written to a txt file.

View the result with `view_registered_fovs.py` (run in napari-env):

`python view_registered_fovs.py <registration txt>`

In [ ]:
from pathlib import Path

from skimage.registration import phase_cross_correlation
import dask.array as da
import numpy as np

def center_pad_nd(fov, target_shape):
    pad_widths = []
    for fov_dim, target_dim in zip(fov.shape, target_shape):
        total_pad = target_dim - fov_dim
        pad_before = total_pad // 2
        pad_widths.append((pad_before, total_pad - pad_before))
    
    return np.pad(fov, pad_widths, mode='constant')

In [ ]:
# ---------------------------------------------------------------- parameters ----
# lowres overview: OME-Zarr group with (t, c, z, y, x) levels 0, 1, 2, ...
LOWRES_ZARR = r"X:\Microglia\E4FAD2\2025_12_01_E4FAD2_Iba635_640_nkt1line_700_MFS4_4x4_10ms_out\fused\2025_12_01_E4FAD2_Iba635_640_nkt1line_700_MFS4_4x4_10ms_fused.zarr"
LOWRES_VOXEL_UM = 4.1012  # level 0, isotropic (not always in the zarr metadata)
LOWRES_CHANNEL = 0
LOWRES_LEVEL = 1  # register at level 1 (2x downsampled, ~8.2 um)

# highres dualview fovs, isotropic 3D arrays (z, y, x)
HIGHRES_VOXEL_UM = 0.512
DWS = {"fov0": r"X:\Microglia\E4FAD2\2025_12_02_E4FAD2_dw_Iba635_640_700_MFS05_1x1_50ms_npy2bdv_ds124_iso0p512_MD0_views.zarr\fused_tp_0_vs_0.zarr\0",
       "fov1": r"X:\Microglia\E4FAD2\2025_12_02_E4FAD2_dw_Iba635_640_700_MFS05_1x1_50ms_1_npy2bdv_ds124_iso0p512_MD0_views.zarr\fused_tp_0_vs_0.zarr\0",
       "fov2": r"X:\Microglia\E4FAD2\2025_12_02_E4FAD2_dw_Iba635_640_700_MFS05_1x1_50ms_2_npy2bdv_ds124_iso0p512_MD0_views.zarr\fused_tp_0_vs_0.zarr\0"}

OUT_TXT = Path(LOWRES_ZARR).parent / "fov_registration.txt"

In [ ]:
# lowres voxel size at the registration level; voxel i of level k is centered at
# (i * 2**k + (2**k - 1) / 2) * LOWRES_VOXEL_UM in lowres level-0 coordinates
level_factor = 2 ** LOWRES_LEVEL
lowres_reg_voxel = LOWRES_VOXEL_UM * level_factor
lowres_reg_offset = (level_factor - 1) / 2 * LOWRES_VOXEL_UM

full_ds = da.from_zarr(str(Path(LOWRES_ZARR) / str(LOWRES_LEVEL)))[0, LOWRES_CHANNEL, ...]
full_ds = full_ds.compute().astype(np.float32)
print('lowres', full_ds.shape, f'{lowres_reg_voxel:.4f} um')

In [ ]:
results = []

for name, dw_path in DWS.items():
    print(name, dw_path)
    dw0 = da.from_zarr(dw_path)
    factor = int(round(lowres_reg_voxel / HIGHRES_VOXEL_UM))
    dw_downsampled = da.coarsen(np.mean, dw0, {0: factor, 1: factor, 2: factor}, trim_excess = True)
    dw_downsampled = dw_downsampled.compute().astype(np.float32)
    padded_fov = center_pad_nd(dw_downsampled, full_ds.shape)
    shift, _, _ = phase_cross_correlation(full_ds, padded_fov, upsample_factor=4)

    # first downsampled voxel position in the lowres registration level (voxel units)
    start = np.array([(F - f) // 2 for F, f in zip(full_ds.shape, dw_downsampled.shape)]) + shift
    # translation of highres voxel 0 (center) in lowres micrometers:
    # center of downsampled voxel 0 is (factor - 1) / 2 highres voxels away from highres voxel 0
    translate_um = start * lowres_reg_voxel + lowres_reg_offset - (factor - 1) / 2 * HIGHRES_VOXEL_UM

    # quality check: correlation between the placed fov and the lowres it lands on
    lo = np.round(start).astype(int)
    hi = lo + np.array(dw_downsampled.shape)
    if np.all(lo >= 0) and np.all(hi <= np.array(full_ds.shape)):
        crop = full_ds[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]
        r = np.corrcoef(crop.ravel(), dw_downsampled.ravel())[0, 1]
    else:
        r = np.nan  # fov placed partly outside the lowres volume
    print('shift', shift, 'translate_um', translate_um.round(2), 'pearson r', round(r, 3))

    results.append(dict(name=name, path=dw_path, voxel_um=HIGHRES_VOXEL_UM,
                        tz=translate_um[0], ty=translate_um[1], tx=translate_um[2],
                        sz=shift[0], sy=shift[1], sx=shift[2], downsample=factor, pearson_r=r))

In [ ]:
# translations are in lowres micrometers (lowres level-0 voxel 0 centered at 0),
# so a highres coordinate maps to the lowres as  lowres_um = highres_voxel * voxel_um + translate_um
columns = ['name', 'path', 'voxel_um', 'tz', 'ty', 'tx', 'sz', 'sy', 'sx', 'downsample', 'pearson_r']
with open(OUT_TXT, 'w') as f:
    f.write(f'# lowres_zarr = {LOWRES_ZARR}\n')
    f.write(f'# lowres_voxel_um = {LOWRES_VOXEL_UM}\n')
    f.write(f'# lowres_channel = {LOWRES_CHANNEL}\n')
    f.write(f'# registration_level = {LOWRES_LEVEL}\n')
    f.write('\t'.join(columns) + '\n')
    for res in results:
        f.write('\t'.join(str(res[c]) for c in columns) + '\n')
print('written', OUT_TXT)